## 1. Get Sample Papers for Chunking

In [ ]:
import sys
from pathlib import Path
import requests

print(f"Python Version: {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}")
print(f"Environment: {sys.executable}")

current_dir = Path.cwd()
if current_dir.name == "tests":
    project_root = current_dir.parent
else:
    project_root = current_dir

print(f"Project Root: {project_root}")

if project_root and (project_root / "src").exists():
    sys.path.insert(0, str(project_root))
else:
    print("Project root not found or src directory missing")
    sys.exit(1)


Python Version: 3.12.11
Environment: /Users/xieqiqi/Learning/LLM/my_first_Rag_project/.venv/bin/python3
Project Root: /Users/xieqiqi/Learning/LLM/my_first_Rag_project


In [8]:
# Get sample papers from database 
from src.db.factory import make_database
from src.models.paper import Paper

print("Fetching sample papers from database")
print("="*40)

database = make_database()

with database.get_session() as session:
    papers = session.query(Paper).filter(
        Paper.raw_text != None,
        Paper.raw_text != ""
    ).limit(3).all()

    if papers:
        print(f"Found {len(papers)} papers with processed text: \n")
        sample_papers = []

        for i, paper in enumerate(papers, 1):
            print(f"{i}.[{paper.arxiv_id}] {paper.title[:60]}...")
            print(f"  Text length: {len(paper.raw_text):,}characters")
            print(f"  Sections available: {'Yes' if paper.sections else 'No'}")

            sample_papers.append({
                "arxiv_id": paper.arxiv_id,
                "title": paper.title,
                "raw_text": paper.raw_text,
                "sections": paper.sections,
                "authors": paper.authors,
                "categories": paper.categories,
                "publish_date": paper.published_date
            })
        test_papers = sample_papers[0]
        print(f"Selected paper for analysis: {test_papers['arxiv_id']}")
    else:
        print("No papers with processed text found.")
        print("Please run the Airflow DAG 'arxiv_paper_ingestion' first.")
        test_paper = None
        sample_papers = []



Fetching sample papers from database
Found 3 papers with processed text: 

1.[2601.08690v1] All Required, In Order: Phase-Level Evaluation for AI-Human ...
  Text length: 63,986characters
  Sections available: Yes
2.[2601.08673v1] Why AI Alignment Failure Is Structural: Learned Human Intera...
  Text length: 74,962characters
  Sections available: Yes
3.[2601.08620v1] ViDoRe V3: A Comprehensive Evaluation of Retrieval Augmented...
  Text length: 104,535characters
  Sections available: Yes
Selected paper for analysis: 2601.08690v1


## 2. Section-based Chunking implementation

In [9]:
import re

def section_based_chunking(text: str, sections_data=None, target_words: int = 600, overlap_words: int=100):
    chunks=[]

    if not sections_data:
        # 
        paragraphs = re.split(r'\n\s*\n', text.strip())
        paragraphs = [p.strip() for p in paragraphs if p.strip()]

        current_chunk = ""
        chunk_index = 0

        for para in paragraphs:
            combined_text = current_chunk + " " + para if current_chunk else para
            if len(combined_text.split()) <= target_words:
                current_chunk = combined_text
            else:
                if current_chunk:
                    chunks.append({
                        'index': chunk_index,
                        'text': current_chunk.strip(),
                        'word_count': len(current_chunk.split()),
                        'section': 'content'
                    })
                    chunk_index += 1
                current_chunk = para

        if current_chunk:
            chunks.append({
                'index': chunk_index,
                'text': current_chunk.strip(),
                'word_count': len(current_chunk.split()),
                'section': 'content'
            })

    else:
        chunk_index = 0

        if isinstance(sections_data, list):
            sections_items = [(item.get('title', f'section_{i}'), item.get('content', '')) for i, item in enumerate(sections_data) if isinstance(item, dict)]        else:
            sections_items = list(sections_data.items())

        for section_name, section_content in sections_items:
            if not section_content or len(str(section_content).strip()) < 50:
                continue

            section_text = str(section_content).strip()
            words = section_text.split()

            if len(words) <= target_words:
                chunks.append({
                    'index': chunk_index,
                    'text': section_text,
                    'word_count': len(words),
                    'section': section_name
                })
                chunk_index += 1
            else:
                start = 0
                while start < len(words):
                    end = start + target_words
                    chunk_words = words[start:end]
                    chunk_text = ' '.join(chunk_words)

                    chunks.append({
                        'index': chunk_index,
                        'text': chunk_text,
                        'word_count': len(chunk_words),
                        'section': section_name,
                        'has_overlap': start > 0
                    })
                    chunk_index += 1
                    start += (target_words - overlap_words)

                    if end >= len(words):
                        break
    return chunks
        
# Text teh chunking system
if test_papers:
    print("section_based chunking results")
    print("=" * 50)

    chunks = section_based_chunking(
        text=test_papers['raw_text'],
        sections_data=test_papers.get('sections'),
        target_words=600,
        overlap_words=100
    )

    print(f"Paper: {test_paper['arxiv_id']}")
    print(f"Original text: {len(test_paper['raw_text'].split()):,} words")
    print(f"Total chunks created: {len(chunks)}")
    print(f"Average chunk size: {sum(c['word_count'] for c in chunks) / len(chunks):.0f} words")

    print("\nSample chunks:")
    for i in range(min(3, len(chunks))):
        chunk = chunks[i]
        print(f"\nChunk {i+1}: {chunk['section']}")
        print(f"  Words: {chunk['word_count']}")
        print(f"  Text preview: {chunk['text'][:150]}...")

    section_counts = {}
    for chunk in chunks:
        section_counts[chunk['section']] = section_counts.get(chunk['section'], 0) + 1

    print("\nSection distribution:")
    print(f"\nChunks per section (top 5):")
    for section, count in list(section_counts.items())[:5]:
        print(f"  {section}: {count} chunks")
        
else:
    print("No test paper available. Please check database connection.")

    

SyntaxError: invalid syntax (2418186391.py, line 41)